In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import json
import os
from pathlib import Path
from typing import Dict, List, Optional
from scipy.stats import pearsonr, spearmanr, wilcoxon, bootstrap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from transformers import AutoTokenizer, RobertaModel, AutoModel
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Paths
BASE = Path("/fs/clip-projects/clip-k12/EMOPAIR")
DATA_DIR = BASE / "Data"
RESULTS_DIR = BASE / "Results"
MODELS_DIR = RESULTS_DIR / "trained models"

# Test all paths:
for path in [BASE, DATA_DIR, RESULTS_DIR, MODELS_DIR]:
    if not path.exists():
        print(f"Warning: {path} does not exist.")

# Device
# if torch.cuda.is_available():
#     device = torch.device("cuda")
#     print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
#     print("Using MPS")
# else:
#     device = torch.device("cpu")
#     print("Using CPU")

# Detect best device available
def get_best_device(cuda_index: int = 0, verbose: bool = True):
    """Detect best device: CUDA (with index) > MPS > CPU"""
    if torch.cuda.is_available():
        if cuda_index < torch.cuda.device_count():
            device = torch.device(f"cuda:{cuda_index}")
            device_name = f"cuda:{cuda_index}"
            if verbose:
                try:
                    gpu_name = torch.cuda.get_device_name(cuda_index)
                    print(f"Using NVIDIA GPU {cuda_index}: {gpu_name}")
                except Exception:
                    print(f"Using NVIDIA GPU {cuda_index}")
        else:
            raise ValueError(
                f"CUDA device {cuda_index} not available. "
                f"Available: 0 to {torch.cuda.device_count()-1}"
            )

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        device_name = "mps"
        if verbose:
            print("Using Apple Silicon GPU via MPS")
    else:
        device = torch.device("cpu")
        device_name = "cpu"
        if verbose:
            print("No GPU found, using CPU")
    return device, device_name
device = get_best_device(cuda_index=1, verbose=True)[0]

# Plot style
sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# Dimension labels
DIM_NAMES = ["Valence", "Arousal", "Dominance"]
DIM_KEYS  = ["valence", "arousal", "dominance"]

# 1 — Model Architectures & Helpers

In [ ]:
# --------------- PERT (fine-tuned RoBERTa for VAD regression) ---------------

class RoBERTaForVAD(nn.Module):
    """PERT: RoBERTa → [CLS] → Dropout → Linear(hidden, 3)  (V, A, D)."""
    def __init__(self, base_model_name: str):
        super().__init__()
        self.base_model = RobertaModel.from_pretrained(base_model_name)
        self.dropout = nn.Dropout(p=0.1)
        self.regressor = nn.Linear(self.base_model.config.hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(outputs.last_hidden_state[:, 0, :])
        return self.regressor(cls)                              # (batch, 3): V, A, D


# --------------- Reward model (pairwise heads) ---------------

def build_reward_model(model_name: str, dropout: float = 0.2) -> nn.Module:
    """Reward model: RoBERTa → [CLS] → Dropout → two linear heads.
    
    valence_head          → Linear(hidden, 1)   pointwise regression
    arousal_dominance_head → Linear(hidden, 2)  pairwise comparison
    """
    encoder = AutoModel.from_pretrained(model_name)
    hidden_size = encoder.config.hidden_size
    model = nn.Module()
    model.encoder = encoder
    model.dropout = nn.Dropout(dropout)
    model.valence_head = nn.Linear(hidden_size, 1)
    model.arousal_dominance_head = nn.Linear(hidden_size, 2)

    def forward(input_ids, attention_mask):
        out = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = model.dropout(out.last_hidden_state[:, 0, :])
        return model.valence_head(h), model.arousal_dominance_head(h)

    model.forward = forward
    return model


# --------------- Prompted-text template (used by PERT) ---------------

PROMPT_SUFFIX = (
    "- Valence: The emotional valence of the context is <mask>. "
    "- Arousal: The level of arousal of the context is <mask>. "
    "- Dominance: The perceived dominance associated with the context is <mask>. "
    "Please predict the missing value for each dimension using the content provided."
)

def prompted(text: str) -> str:
    return f"Context: '{text}' {PROMPT_SUFFIX}"


# --------------- Generic scoring helpers ---------------

@torch.no_grad()
def score_pert(model, tokenizer, texts, batch_size=64, max_length=256):
    """Score texts with PERT model → dict of {valence, arousal, dominance} arrays."""
    model.eval()
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i+batch_size], max_length=max_length,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        out = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        preds.append(out.cpu().numpy())
    preds = np.concatenate(preds, axis=0)
    return {k: preds[:, i] for i, k in enumerate(DIM_KEYS)}


@torch.no_grad()
def score_reward(model, tokenizer, texts, batch_size=64, max_length=256):
    """Score texts with reward model → dict of {valence, arousal, dominance} arrays."""
    model.eval()
    sv, sa, sd = [], [], []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i+batch_size], max_length=max_length,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        v, ad = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        sv.extend(v[:, 0].cpu().numpy())
        sa.extend(ad[:, 0].cpu().numpy())
        sd.extend(ad[:, 1].cpu().numpy())
    return {"valence": np.array(sv), "arousal": np.array(sa), "dominance": np.array(sd)}

print("Architectures and helpers defined.")

# 2 — Load All Four Models

In [ ]:
# ---- PERT tokenizers (robust loader across tokenizer format/version issues) ----
def load_roberta_tokenizer(model_dir, base_model_name):
    try:
        return AutoTokenizer.from_pretrained(model_dir, use_fast=True)
    except Exception as e:
        print(f"Tokenizer load from checkpoint failed ({type(e).__name__}); using base tokenizer '{base_model_name}'.")
        return AutoTokenizer.from_pretrained(base_model_name, use_fast=True)

# ---- PERT-EmoPair ----
pert_emopair_dir = MODELS_DIR / "03-09-26_15-28_pertuned_VAD_roberta-large_emopair" 
ckpt = torch.load(pert_emopair_dir / "pytorch_model.bin", map_location=device, weights_only=False)
pert_emopair = RoBERTaForVAD(ckpt["model_config"]["base_model_name"])
pert_emopair.load_state_dict(ckpt["model_state_dict"])
pert_emopair.to(device).eval()
tok_pert_emopair = load_roberta_tokenizer(
    pert_emopair_dir,
    ckpt["model_config"]["base_model_name"],
)
print("✓ PERT-EmoPair loaded")

# ---- PERT-EmoBank ----
pert_emobank_dir = MODELS_DIR / "emopair_vs_emobank" / "01-29-26_16-01_pertuned_VAD_roberta-large_emobank"
ckpt = torch.load(pert_emobank_dir / "pytorch_model.bin", map_location=device, weights_only=False)
pert_emobank = RoBERTaForVAD(ckpt["model_config"]["base_model_name"])
pert_emobank.load_state_dict(ckpt["model_state_dict"])
pert_emobank.to(device).eval()
tok_pert_emobank = load_roberta_tokenizer(
    pert_emobank_dir,
    ckpt["model_config"]["base_model_name"],
)
print("✓ PERT-EmoBank loaded")

# ---- Reward-EmoPair ----
rw_emopair_path = MODELS_DIR / "emopair_reward_model_20260310.pth"
rw_emopair_ckpt = torch.load(rw_emopair_path, map_location=device, weights_only=False)
rw_emopair_cfg = rw_emopair_ckpt["config"]
rw_emopair = build_reward_model(rw_emopair_cfg["model_name"], dropout=0.2)
rw_emopair.load_state_dict(rw_emopair_ckpt["model_state_dict"])
rw_emopair.to(device).eval()
tok_rw_emopair = AutoTokenizer.from_pretrained(rw_emopair_cfg["model_name"])
print("✓ Reward-EmoPair loaded")

# ---- Reward-EmoBank ----
rw_emobank_path = MODELS_DIR / "emobank_reward_model_20260311.pth"
rw_emobank_ckpt = torch.load(rw_emobank_path, map_location=device, weights_only=False)
rw_emobank_cfg = rw_emobank_ckpt["config"]
rw_emobank = build_reward_model(rw_emobank_cfg["model_name"], dropout=0.2)
rw_emobank.load_state_dict(rw_emobank_ckpt["model_state_dict"])
rw_emobank.to(device).eval()
tok_rw_emobank = AutoTokenizer.from_pretrained(rw_emobank_cfg["model_name"])
print("✓ Reward-EmoBank loaded")

# 3 — Prepare Test Data & Score All Models

In [ ]:
# Load data
emopair = pd.read_csv(RESULTS_DIR / "emopair.csv")
emopair_pairs = pd.read_csv(RESULTS_DIR / "emopair_pairs_w_breakdowns.csv")

# Test-set items (single texts with gold labels)
test_items = emopair[emopair["split"] == "test"].dropna(subset=["text", "V"]).copy()
test_texts = test_items["text"].tolist()
test_texts_prompted = [prompted(t) for t in test_texts]

# Gold targets
gold_V = test_items["V"].values.astype(np.float32)
gold_A = test_items["A"].values.astype(np.float32)
gold_D = test_items["D"].values.astype(np.float32)
gold_A_bt = test_items["A_BT_score_full"].values.astype(np.float32)
gold_D_bt = test_items["D_BT_score_full"].values.astype(np.float32)

print(f"Test items:  {len(test_items)}")

# Test-set pairs (pairwise decisions)
test_pairs = emopair_pairs[
    (emopair_pairs["item1_split"] == "test") &
    (emopair_pairs["item2_split"] == "test") &
    emopair_pairs["item1_text"].notna() &
    emopair_pairs["item2_text"].notna() &
    emopair_pairs["A_decision"].isin(["Text1", "Text2"]) &
    emopair_pairs["D_decision"].isin(["Text1", "Text2"])
].copy()
print(f"Test pairs:  {len(test_pairs)}")


In [ ]:
# Score every test text with all four models
print("Scoring with PERT-EmoPair …")
pred_pert_emopair = score_pert(pert_emopair, tok_pert_emopair, test_texts_prompted)

print("Scoring with PERT-EmoBank …")
pred_pert_emobank = score_pert(pert_emobank, tok_pert_emobank, test_texts_prompted)

print("Scoring with Reward-EmoPair (raw) …")
raw_rw_emopair = score_reward(rw_emopair, tok_rw_emopair, test_texts)

print("Scoring with Reward-EmoBank (raw) …")
raw_rw_emobank = score_reward(rw_emobank, tok_rw_emobank, test_texts)

# Convenience dict: model_name -> score dict
MODELS = {
    "PERT-EmoPair": pred_pert_emopair,
    "PERT-EmoBank": pred_pert_emobank,
    "Reward-EmoPair": raw_rw_emopair,
    "Reward-EmoBank": raw_rw_emobank,
}
MODEL_NAMES = list(MODELS.keys())
print("All scoring complete.")

# 4 — Pointwise Regression: Valence (MAE · Pearson *r*)

Valence is the one dimension where the older EmoBank-style 1–5 score is directly available and reasonably comparable across settings, so this section is mostly a calibration check rather than the decisive test of label quality. If EmoPair supervision were broadly weak, its models would fall apart here too; instead, both PERT models remain very strong and nearly tied in correlation. The sharper data-quality contrast appears in Arousal and Dominance, where EmoPair contributes direct paired-comparison supervision and EmoBank does not.

In [ ]:
rows = []
for name in MODEL_NAMES:
    pv = MODELS[name]["valence"]
    rows.append({
        "Model": name,
        "MAE":     mean_absolute_error(gold_V, pv),
        "RMSE":    np.sqrt(mean_squared_error(gold_V, pv)),
        'R^2': r2_score(gold_V, pv),
        "Pearson r": pearsonr(gold_V, pv)[0],
        "Spearman ρ": spearmanr(gold_V, pv).statistic,
    })

valence_df = pd.DataFrame(rows).set_index("Model")
valence_df.style.format("{:.4f}").highlight_min(
    subset=["MAE", "RMSE"], props="font-weight:bold"
).highlight_max(
    subset=["Pearson r", "Spearman ρ"], props="font-weight:bold"
)

# 5 — Arousal & Dominance: Correlation with Bradley-Terry Scores

For Arousal and Dominance, the most faithful target inside EmoPair is the Bradley-Terry (BT) score derived from pairwise judgments. This is the first direct test of whether the paired-comparison framework is yielding a cleaner signal than the older EmoBank score averages: if EmoPair labels are better structured, models trained on EmoPair should align more strongly with these BT targets than models trained on EmoBank.

In [ ]:
# Mask to items that have valid BT scores
bt_mask = np.isfinite(gold_A_bt) & np.isfinite(gold_D_bt)

rows = []
for name in MODEL_NAMES:
    raw = MODELS[name]
    pa = raw["arousal"][bt_mask]
    pd_ = raw["dominance"][bt_mask]
    rows.append({
        "Model": name,
        # BT score correlations (pairwise-derived)
        "A BT Pearson r":    pearsonr(gold_A_bt[bt_mask], pa)[0],
        "A BT Spearman ρ":   spearmanr(gold_A_bt[bt_mask], pa).statistic,
        "D BT Pearson r":    pearsonr(gold_D_bt[bt_mask], pd_)[0],
        "D BT Spearman ρ":   spearmanr(gold_D_bt[bt_mask], pd_).statistic,
        # Original 1–5 score correlations (raw annotator averages)
        "A orig Pearson r":  pearsonr(gold_A, raw["arousal"])[0],
        "A orig Spearman ρ": spearmanr(gold_A, raw["arousal"]).statistic,
        "D orig Pearson r":  pearsonr(gold_D, raw["dominance"])[0],
        "D orig Spearman ρ": spearmanr(gold_D, raw["dominance"]).statistic,
    })

bt_df = pd.DataFrame(rows).set_index("Model")
bt_df.style.format("{:.4f}").highlight_max(props="font-weight:bold")


# 6 — Pairwise Accuracy on Test Pairs

This is the most direct behavioral evaluation of the annotation schemes. Rather than asking whether a model reproduces an old numeric scale, we ask whether it agrees with human comparative judgments on held-out pairs. Strong performance here is evidence that the training data captures the relative ordering humans actually use, which is exactly the promise of the EmoPair paired-comparison design.

For **Valence** the human label comes from gold V scores (the item with higher V wins). For **Arousal** and **Dominance** the human label comes from the pairwise annotation decisions (`A_decision`, `D_decision`).

In [ ]:
# Build a lookup: item_id → index in test_items for fast access
test_id_to_idx = {iid: i for i, iid in enumerate(test_items["item_id"].values)}

def pairwise_accuracy_for_model(model_name: str, dim: str):
    """Compute pairwise accuracy for a model on a given dimension.
    
    Returns (n_correct, n_total, accuracy).
    """
    raw = MODELS[model_name]
    
    correct, total = 0, 0
    for _, row in test_pairs.iterrows():
        i1 = test_id_to_idx.get(row["item1"])
        i2 = test_id_to_idx.get(row["item2"])
        if i1 is None or i2 is None:
            continue
        
        s1 = raw[dim][i1]
        s2 = raw[dim][i2]
        if s1 == s2:
            continue  # skip ties in model output
        model_says_1 = s1 > s2
        
        if dim == "valence":
            v1, v2 = gold_V[i1], gold_V[i2]
            if v1 == v2:
                continue
            human_says_1 = v1 > v2
        elif dim == "arousal":
            dec = row.get("A_decision")
            if dec not in ("Text1", "Text2"):
                continue
            human_says_1 = (dec == "Text1")
        else:  # dominance
            dec = row.get("D_decision")
            if dec not in ("Text1", "Text2"):
                continue
            human_says_1 = (dec == "Text1")
        
        total += 1
        if model_says_1 == human_says_1:
            correct += 1
    
    return correct, total, correct / total if total else float("nan")


rows = []
for name in MODEL_NAMES:
    row = {"Model": name}
    for dim, label in zip(DIM_KEYS, DIM_NAMES):
        c, t, acc = pairwise_accuracy_for_model(name, dim)
        row[f"{label} Acc"] = acc
        row[f"{label} n"] = t
    rows.append(row)

pw_df = pd.DataFrame(rows).set_index("Model")
pw_df.style.format({c: "{:.2%}" for c in pw_df.columns if "Acc" in c}).format(
    {c: "{:.0f}" for c in pw_df.columns if " n" in c}
).highlight_max(subset=[c for c in pw_df.columns if "Acc" in c], props="font-weight:bold")

In [ ]:
# Build a lookup: item_id -> index in test_items for fast access
test_id_to_idx = {
    item_id: idx for idx, item_id in enumerate(test_items["item_id"].values)
}

GOLD_SCORES = {
    "valence": gold_V,
    "arousal": gold_A,
    "dominance": gold_D,
}

DECISION_COLS = {
    "arousal": "A_decision",
    "dominance": "D_decision",
}

def pairwise_accuracy_for_model(
    model_name: str,
    dim: str,
    label_source: str = "emopair",
    tie_eps: float = 0.0,
):
    """
    label_source:
        "emopair" -> valence from gold score order; arousal/dominance from A_decision/D_decision
        "emobank" -> valence/arousal/dominance from original item-level score order
    """
    raw_scores = MODELS[model_name]
    gold_scores = GOLD_SCORES[dim]

    correct = 0
    total = 0

    for _, row in test_pairs.iterrows():
        idx1 = test_id_to_idx.get(row["item1"])
        idx2 = test_id_to_idx.get(row["item2"])
        if idx1 is None or idx2 is None:
            continue

        pred1 = raw_scores[dim][idx1]
        pred2 = raw_scores[dim][idx2]
        if (
            not np.isfinite(pred1)
            or not np.isfinite(pred2)
            or abs(pred1 - pred2) <= tie_eps
        ):
            continue

        model_says_text1 = pred1 > pred2

        use_score_order = (dim == "valence") or (label_source == "emobank")
        if use_score_order:
            gold1 = gold_scores[idx1]
            gold2 = gold_scores[idx2]
            if (
                not np.isfinite(gold1)
                or not np.isfinite(gold2)
                or abs(gold1 - gold2) <= tie_eps
            ):
                continue
            gold_says_text1 = gold1 > gold2
        else:
            decision = row.get(DECISION_COLS[dim])
            if decision not in ("Text1", "Text2"):
                continue
            gold_says_text1 = decision == "Text1"

        total += 1
        if model_says_text1 == gold_says_text1:
            correct += 1

    accuracy = correct / total if total else float("nan")
    return correct, total, accuracy

rows = []
for name in MODEL_NAMES:
    row = {"Model": name}

    # Existing metric: valence score-order, A/D direct pairwise decisions
    for dim, label in zip(DIM_KEYS, DIM_NAMES):
        _, total, acc = pairwise_accuracy_for_model(
            name,
            dim,
            label_source="emopair",
        )
        row[f"{label} Acc"] = acc
        row[f"{label} n"] = total

    # Additional metric: A/D pairwise accuracy from original EmoBank score order
    for dim, label in [("arousal", "Arousal"), ("dominance", "Dominance")]:
        _, total, acc = pairwise_accuracy_for_model(
            name,
            dim,
            label_source="emobank",
        )
        row[f"{label} EmoBank Acc"] = acc
        row[f"{label} EmoBank n"] = total

    rows.append(row)

pw_df = pd.DataFrame(rows).set_index("Model")
pw_df.style.format(
    {col: "{:.2%}" for col in pw_df.columns if "Acc" in col}
).format(
    {col: "{:.0f}" for col in pw_df.columns if " n" in col}
).highlight_max(
    subset=[col for col in pw_df.columns if "Acc" in col],
    props="font-weight:bold",
)

# 7 — Statistical Significance Tests

These tests separate real differences from noise. They matter for the data-quality question because a consistent gap in favor of EmoPair-trained models on pairwise targets would suggest that the paired-comparison labels are not just different from EmoBank, but measurably more informative for the judgments we care about.

We compare the top two models for each metric using:
- **Bootstrap 95% CI** on the difference in Pearson *r* (10,000 resamples)
- **Wilcoxon signed-rank** on absolute-error distributions (Valence only)

In [ ]:
rng = np.random.default_rng(42)

def bootstrap_pearson_diff(gold, pred_a, pred_b, n_boot=10_000, ci=0.95):
    """Bootstrap CI for Pearson(gold, pred_a) − Pearson(gold, pred_b)."""
    n = len(gold)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ra = pearsonr(gold[idx], pred_a[idx])[0]
        rb = pearsonr(gold[idx], pred_b[idx])[0]
        diffs[b] = ra - rb
    lo = np.percentile(diffs, (1 - ci) / 2 * 100)
    hi = np.percentile(diffs, (1 + ci) / 2 * 100)
    return np.mean(diffs), lo, hi

# --- Valence: Wilcoxon on |error| between best two models ---
abs_errors = {name: np.abs(gold_V - MODELS[name]["valence"]) for name in MODEL_NAMES}
best_two_v = sorted(MODEL_NAMES, key=lambda n: abs_errors[n].mean())[:2]
stat_w, p_w = wilcoxon(abs_errors[best_two_v[0]], abs_errors[best_two_v[1]])
print(f"Valence |error| Wilcoxon: {best_two_v[0]} vs {best_two_v[1]}")
print(f"  statistic = {stat_w:.1f},  p = {p_w:.4e}")

# --- V Pearson bootstrap ---
best_two_vp = sorted(
    MODEL_NAMES,
    key=lambda n: pearsonr(gold_V, MODELS[n]["valence"])[0],
    reverse=True
)[:2]
diff_mean, ci_lo, ci_hi = bootstrap_pearson_diff(
    gold_V,
    MODELS[best_two_vp[0]]["valence"],
    MODELS[best_two_vp[1]]["valence"],
)
print(f"\nValence Pearson Δ: {best_two_vp[0]} − {best_two_vp[1]}")
print(f"  Δ = {diff_mean:.4f}  95 % CI [{ci_lo:.4f}, {ci_hi:.4f}]")

# --- Arousal BT Pearson bootstrap ---
best_two_a = sorted(
    MODEL_NAMES,
    key=lambda n: pearsonr(gold_A_bt[bt_mask], MODELS[n]["arousal"][bt_mask])[0],
    reverse=True
)[:2]
diff_mean_a, ci_lo_a, ci_hi_a = bootstrap_pearson_diff(
    gold_A_bt[bt_mask],
    MODELS[best_two_a[0]]["arousal"][bt_mask],
    MODELS[best_two_a[1]]["arousal"][bt_mask],
)
print(f"\nArousal BT Pearson Δ: {best_two_a[0]} − {best_two_a[1]}")
print(f"  Δ = {diff_mean_a:.4f}  95 % CI [{ci_lo_a:.4f}, {ci_hi_a:.4f}]")

# --- Dominance BT Pearson bootstrap ---
best_two_d = sorted(
    MODEL_NAMES,
    key=lambda n: pearsonr(gold_D_bt[bt_mask], MODELS[n]["dominance"][bt_mask])[0],
    reverse=True
)[:2]
diff_mean_d, ci_lo_d, ci_hi_d = bootstrap_pearson_diff(
    gold_D_bt[bt_mask],
    MODELS[best_two_d[0]]["dominance"][bt_mask],
    MODELS[best_two_d[1]]["dominance"][bt_mask],
)
print(f"\nDominance BT Pearson Δ: {best_two_d[0]} − {best_two_d[1]}")
print(f"  Δ = {diff_mean_d:.4f}  95 % CI [{ci_lo_d:.4f}, {ci_hi_d:.4f}]")

# --- Arousal Original Pearson bootstrap ---
best_two_a_orig = sorted(
    MODEL_NAMES,
    key=lambda n: pearsonr(gold_A, MODELS[n]["arousal"])[0],
    reverse=True
)[:2]
diff_mean_a_orig, ci_lo_a_orig, ci_hi_a_orig = bootstrap_pearson_diff(
    gold_A,
    MODELS[best_two_a_orig[0]]["arousal"],
    MODELS[best_two_a_orig[1]]["arousal"],
)
print(f"\nArousal Orig Pearson Δ: {best_two_a_orig[0]} − {best_two_a_orig[1]}")
print(f"  Δ = {diff_mean_a_orig:.4f}  95 % CI [{ci_lo_a_orig:.4f}, {ci_hi_a_orig:.4f}]")

# --- Dominance Original Pearson bootstrap ---
best_two_d_orig = sorted(
    MODEL_NAMES,
    key=lambda n: pearsonr(gold_D, MODELS[n]["dominance"])[0],
    reverse=True
)[:2]
diff_mean_d_orig, ci_lo_d_orig, ci_hi_d_orig = bootstrap_pearson_diff(
    gold_D,
    MODELS[best_two_d_orig[0]]["dominance"],
    MODELS[best_two_d_orig[1]]["dominance"],
)
print(f"\nDominance Orig Pearson Δ: {best_two_d_orig[0]} − {best_two_d_orig[1]}")
print(f"  Δ = {diff_mean_d_orig:.4f}  95 % CI [{ci_lo_d_orig:.4f}, {ci_hi_d_orig:.4f}]")

## Updated interpretation

1. **Valence is not where the two datasets separate much.** PERT-EmoBank has the lowest valence error (MAE = 0.1442 vs. 0.1527 for PERT-EmoPair), and that MAE gap is significant (Wilcoxon $p = 2.59 \times 10^{-4}$). But the correlation gap is effectively zero: PERT-EmoPair has Pearson $r = 0.8338$ vs. 0.8326 for PERT-EmoBank, with bootstrap $\Delta r = 0.0011$ and a CI that crosses zero. So the older EmoBank scores help calibration slightly, but they do not reveal a meaningfully better valence signal.

2. **On EmoPair-defined Arousal and Dominance targets, EmoPair-trained models win clearly.** Reward-EmoPair and PERT-EmoPair are far ahead of the EmoBank-trained models on BT correlation and pairwise accuracy. For Arousal BT, Reward-EmoPair reaches $r = 0.8706$ and PERT-EmoPair reaches 0.8616, versus only 0.6273 and 0.6144 for the EmoBank models. For Dominance BT, PERT-EmoPair reaches $r = 0.7313$ and Reward-EmoPair 0.7263, versus 0.4423 and 0.3177 for the EmoBank models. The same pattern appears in pairwise accuracy: Arousal is 85.37% and 84.34% for EmoPair-trained models versus 73.99% and 72.82% for EmoBank-trained models; Dominance is 77.03% and 76.96% versus 64.93% and 61.33%.

3. **Those gaps are large in practice even when the top two EmoPair models are statistically tied.** The significance tests show no meaningful difference between Reward-EmoPair and PERT-EmoPair on BT correlation: Arousal $\Delta r = 0.0090$, 95% CI [-0.0032, 0.0216]; Dominance $\Delta r = 0.0050$, 95% CI [-0.0135, 0.0236]. That means the key result is not architecture. It is that either architecture works much better once it is trained on EmoPair labels.

4. **When evaluation switches back to the original EmoBank score averages, the ranking flips.** PERT-EmoBank is best on the original-score correlations for both Arousal ($r = 0.5898$) and Dominance ($r = 0.5317$), and its advantage over Reward-EmoBank is significant: Arousal $\Delta r = 0.1042$, 95% CI [0.0729, 0.1363]; Dominance $\Delta r = 0.1416$, 95% CI [0.0910, 0.1926]. This does **not** mean EmoBank is the better supervision source overall. It means EmoBank-trained models are better at reproducing the older 1–5 score scale that EmoBank itself defines.

5. **Taken together, the evidence favors EmoPair as the higher-quality source for comparative affect judgments.** The notebook results show that EmoPair-trained models generalize much better to held-out human pairwise judgments and BT scores, especially on Arousal and Dominance where annotation difficulty is highest. Separately, we also have reliability analyses outside this notebook showing higher annotator agreement for EmoPair under the paired-comparison framework. Those two pieces of evidence point in the same direction: the EmoPair labels appear more internally consistent and more behaviorally faithful, while the older EmoBank averages appear more entangled with scale-use noise and annotator calibration differences.

6. **The cleanest conclusion is about fitness for purpose.** If the goal is compatibility with historical EmoBank-style absolute scores, EmoBank remains useful, especially for Valence. If the goal is to model how humans comparatively judge emotional content, the evidence here strongly supports EmoPair as the better data source.

# 8 — Scatter Plots: Predicted vs. Gold (Valence)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharex=True, sharey=True)
colors = sns.color_palette("Set2", 4)

for ax, name, color in zip(axes, MODEL_NAMES, colors):
    pv = MODELS[name]["valence"]
    r = pearsonr(gold_V, pv)[0]
    mae = mean_absolute_error(gold_V, pv)
    ax.scatter(gold_V, pv, s=8, alpha=0.35, color=color, edgecolors="none")
    ax.plot([1, 5], [1, 5], "k--", lw=0.8, alpha=0.5)
    ax.set_title(f"{name}\nr = {r:.3f}  MAE = {mae:.3f}", fontsize=10)
    ax.set_xlabel("Gold Valence")
    ax.set_aspect("equal")

axes[0].set_ylabel("Predicted Valence")
fig.suptitle("Valence: Predicted vs. Gold (Test Set)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "visuals" / "valence_scatter_comparison.pdf", bbox_inches="tight")
plt.show()

# 9 — Scatter Plots: Model Scores vs. BT Scores (Arousal & Dominance)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7.5))
colors = sns.color_palette("Set2", 4)

for col, (name, color) in enumerate(zip(MODEL_NAMES, colors)):
    raw = MODELS[name]
    for row_idx, (dim, gold_bt, label) in enumerate([
        ("arousal", gold_A_bt, "Arousal"),
        ("dominance", gold_D_bt, "Dominance"),
    ]):
        ax = axes[row_idx, col]
        pred = raw[dim][bt_mask]
        gt = gold_bt[bt_mask]
        r = pearsonr(gt, pred)[0]
        ax.scatter(gt, pred, s=8, alpha=0.3, color=color, edgecolors="none")
        # trend line
        z = np.polyfit(gt, pred, 1)
        xs = np.linspace(gt.min(), gt.max(), 50)
        ax.plot(xs, np.polyval(z, xs), "k--", lw=0.8, alpha=0.6)
        ax.set_title(f"{name}\nr = {r:.3f}", fontsize=9)
        if col == 0:
            ax.set_ylabel(f"Predicted {label}")
        if row_idx == 1:
            ax.set_xlabel("BT Score")

fig.suptitle("Arousal & Dominance: Model Scores vs. BT Scores (Test Set)", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "visuals" / "arousal_dominance_bt_scatter.pdf", bbox_inches="tight")
plt.show()

# 10 — Valence Error Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = sns.color_palette("Set2", 4)

# (a) Signed error histogram
ax = axes[0]
for name, color in zip(MODEL_NAMES, colors):
    err = MODELS[name]["valence"] - gold_V
    ax.hist(err, bins=50, alpha=0.45, label=name, color=color, density=True)
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_xlabel("Signed Error (Predicted − Gold)")
ax.set_ylabel("Density")
ax.set_title("Valence: Signed Error Distribution")
ax.legend(fontsize=8)

# (b) Absolute error box-plot
ax = axes[1]
err_data = [np.abs(gold_V - MODELS[n]["valence"]) for n in MODEL_NAMES]
bp = ax.boxplot(err_data, labels=[n.replace("-", "\n") for n in MODEL_NAMES],
                patch_artist=True, showfliers=False, widths=0.5)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("|Error|")
ax.set_title("Valence: Absolute Error")

fig.tight_layout()
fig.savefig(RESULTS_DIR / "visuals" / "valence_error_distributions.pdf", bbox_inches="tight")
plt.show()

# 11 — Grouped Bar Chart: All Metrics at a Glance

In [ ]:

# Gather key metrics into a single dataframe for plotting
bar_rows = []
for name in MODEL_NAMES:
    pv = MODELS[name]["valence"]
    raw = MODELS[name]
    bar_rows.append({
        "Model": name,
        "V Pearson r":      pearsonr(gold_V, pv)[0],
        "A BT r":           pearsonr(gold_A_bt[bt_mask], raw["arousal"][bt_mask])[0],
        "D BT r":           pearsonr(gold_D_bt[bt_mask], raw["dominance"][bt_mask])[0],
        "A orig r":         pearsonr(gold_A, raw["arousal"])[0],
        "D orig r":         pearsonr(gold_D, raw["dominance"])[0],
    })
bar_df = pd.DataFrame(bar_rows).set_index("Model")

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(bar_df.columns))
width = 0.16
colors = sns.color_palette("Set2", 4)

for i, (name, color) in enumerate(zip(MODEL_NAMES, colors)):
    vals = bar_df.loc[name].values
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, vals, width, label=name, color=color, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                f"{v:.3f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(bar_df.columns, fontsize=10)
ax.set_ylabel("Pearson r")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, loc="lower right")
ax.set_title("Pearson Correlation: All Dimensions (Test Set)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "visuals" / "pearson_bar_comparison.pdf", bbox_inches="tight")
plt.show()


# 12 — Training Curves

In [ ]:
# Load training histories
with open(pert_emopair_dir / "epoch_history.json") as f:
    hist_pert_emopair = pd.DataFrame(json.load(f))
with open(pert_emobank_dir / "epoch_history.json") as f:
    hist_pert_emobank = pd.DataFrame(json.load(f))

hist_rw_emopair = pd.read_csv(MODELS_DIR / "emopair_reward_model_training_history_20260310.csv")
hist_rw_emobank = pd.read_csv(MODELS_DIR / "emobank_reward_model_training_history_20260308.csv")

histories = {
    "PERT-EmoPair":   hist_pert_emopair,
    "PERT-EmoBank":   hist_pert_emobank,
    "Reward-EmoPair": hist_rw_emopair,
    "Reward-EmoBank": hist_rw_emobank,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = sns.color_palette("Set2", 4)

# (a) Training loss
ax = axes[0]
for (name, h), color in zip(histories.items(), colors):
    ax.plot(h["epoch"], h["train_loss"], label=name, color=color, lw=1.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.set_title("Training Loss")
ax.legend(fontsize=8)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# (b) Eval loss
ax = axes[1]
for (name, h), color in zip(histories.items(), colors):
    col = "val_loss" if "val_loss" in h.columns else "eval_loss"
    ax.plot(h["epoch"], h[col], label=name, color=color, lw=1.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Validation / Eval Loss")
ax.set_title("Validation Loss")
ax.legend(fontsize=8)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

fig.suptitle("Training Curves", fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "visuals" / "training_curves.pdf", bbox_inches="tight")
plt.show()

# 13 — Pairwise Accuracy Bar Chart

In [ ]:
acc_cols = [c for c in pw_df.columns if "Acc" in c]
acc_plot = pw_df[acc_cols].copy()
acc_plot.columns = [c.replace(" Acc", "") for c in acc_cols]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(acc_plot.columns))
width = 0.18
colors = sns.color_palette("Set2", 4)

for i, (name, color) in enumerate(zip(MODEL_NAMES, colors)):
    vals = acc_plot.loc[name].values
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, vals * 100, width, label=name, color=color, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v:.1%}", ha="center", va="bottom", fontsize=7)

ax.axhline(50, color="grey", ls=":", lw=0.8, label="Chance")
ax.set_xticks(x)
ax.set_xticklabels(acc_plot.columns, fontsize=11)
ax.set_ylabel("Pairwise Accuracy (%)")
ax.set_ylim(40, 100)
ax.legend(fontsize=9, loc="lower right")
ax.set_title("Pairwise Ranking Accuracy (Test Set)")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "visuals" / "pairwise_accuracy_bar.pdf", bbox_inches="tight")
plt.show()

# 14 — Consolidated Summary Table (LaTeX-ready)

This is the compact publication table for the full comparison. It is meant to foreground the main substantive result: models trained on EmoPair lead on the pairwise targets that most directly reflect comparative human judgments, while EmoBank-trained models retain an advantage mainly on the legacy original-score scale. The export cell below writes a LaTeX version with bolded column winners and a caption that states that interpretation directly.

In [ ]:
summary_rows = []
for name in MODEL_NAMES:
    pv = MODELS[name]["valence"]
    raw = MODELS[name]
    pa = raw["arousal"][bt_mask]
    pd_ = raw["dominance"][bt_mask]

    # Pairwise accuracy (recompute inline)
    acc_v = pairwise_accuracy_for_model(name, "valence")[2]
    acc_a = pairwise_accuracy_for_model(name, "arousal")[2]
    acc_d = pairwise_accuracy_for_model(name, "dominance")[2]

    summary_rows.append({
        "Model": name,
        # Valence pointwise
        "V MAE ↓":            mean_absolute_error(gold_V, pv),
        "V Pearson r ↑":      pearsonr(gold_V, pv)[0],
        # BT correlations (pairwise-derived scores)
        "A BT r ↑":           pearsonr(gold_A_bt[bt_mask], pa)[0],
        "D BT r ↑":           pearsonr(gold_D_bt[bt_mask], pd_)[0],
        # Original 1–5 score correlations
        "A orig r ↑":         pearsonr(gold_A, raw["arousal"])[0],
        "D orig r ↑":         pearsonr(gold_D, raw["dominance"])[0],
        # Pairwise accuracy
        "V Pair Acc ↑":       acc_v,
        "A Pair Acc ↑":       acc_a,
        "D Pair Acc ↑":       acc_d,
    })

summary = pd.DataFrame(summary_rows).set_index("Model")

# Display with highlighting
styled = summary.style.format("{:.4f}").highlight_min(
    subset=[c for c in summary.columns if "↓" in c], props="font-weight:bold"
).highlight_max(
    subset=[c for c in summary.columns if "↑" in c], props="font-weight:bold"
)
styled


In [ ]:
# Export LaTeX table with bolded column winners
latex_table = summary.copy()

for col in latex_table.columns:
    if "↓" in col:
        best_mask = np.isclose(summary[col], summary[col].min())
    else:
        best_mask = np.isclose(summary[col], summary[col].max())

    latex_table[col] = summary[col].map(lambda value: f"{value:.4f}")
    latex_table.loc[best_mask, col] = summary.loc[best_mask, col].map(
        lambda value: rf"\textbf{{{value:.4f}}}"
    )

latex = latex_table.to_latex(
    escape=False,
    caption=(
        "Comparison of four RoBERTa-large models on the EmoPair test set. "
        "Models trained on EmoPair lead on the pairwise evaluation targets "
        "(Bradley-Terry correlations and pairwise accuracy), whereas "
        "EmoBank-trained models are strongest mainly on the legacy original-score "
        "correlations and slightly lower valence error. This pattern supports the "
        "interpretation that paired-comparison supervision in EmoPair is better "
        "aligned with comparative affect judgments. "
        r"BT $r$ = Pearson correlation with Bradley-Terry scores; "
        r"orig $r$ = Pearson correlation with original 1--5 score averages; "
        r"$\uparrow$ = higher is better; $\downarrow$ = lower is better."
    ),
    label="tab:model_comparison",
)

print(latex)

with open(RESULTS_DIR / "visuals" / "model_comparison_table.tex", "w") as f:
    f.write(latex)
print("\nLaTeX table saved to Results/visuals/model_comparison_table.tex")

# 15 — Radar Chart: Per-Model Profile

In [ ]:
from matplotlib.transforms import ScaledTranslation

# Radar chart: Pearson r across 3 dims (BT) + orig A/D r + pairwise accuracy for 3 dims = 8 axes
radar_metrics = [
    "V Pearson r ↑", "A BT r ↑", "D BT r ↑",
    "A orig r ↑", "D orig r ↑",
    "V Pair Acc ↑", "A Pair Acc ↑", "D Pair Acc ↑",
]
radar_labels = [
    "Valence \n Pearson r", "Arousal BT r", "Dominance BT r",
    "Arousal orig r", "Dominance \n orig r",
    "Valence Pair Acc", "Arousal Pair Acc", "Dominance Pair Acc",
]
n_metrics = len(radar_metrics)

angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = sns.color_palette("Set2", 4)

for name, color in zip(MODEL_NAMES, colors):
    vals = [summary.loc[name, m] for m in radar_metrics]
    vals += vals[:1]
    ax.plot(angles, vals, "-o", color=color, label=name, lw=1.8, markersize=5, zorder=2)
    ax.fill(angles, vals, color=color, alpha=0.08, zorder=1)

gridlines, theta_labels = ax.set_thetagrids(
    np.degrees(angles[:-1]),
    radar_labels,
    fontsize=12,
 )
ax.tick_params(axis="x", pad=12)

# Per-label extra offsets in points: (x_offset, y_offset).
label_offsets = {
    "Valence \n Pearson r": (10, 6),
    "Arousal BT r": (0, 4),
    "Dominance BT r": (0, -12),
    "Arousal orig r": (-4, 8),
    "Dominance \n orig r": (-6, 18),
    "Valence Pair Acc": (-20, -10),
    "Arousal Pair Acc": (0, 10),
    "Dominance Pair Acc": (24, -4),
}

for label in theta_labels:
    label.set_zorder(10)
    label.set_bbox(dict(facecolor="white", edgecolor="none", alpha=0.85, pad=0.2))
    dx, dy = label_offsets.get(label.get_text(), (0, 0))
    label.set_transform(
        label.get_transform()
        + ScaledTranslation(dx / 72, dy / 72, fig.dpi_scale_trans)
    )

ax.set_ylim(0, 1)
ax.set_title("Model Profiles \n(Test Set)", fontsize=15, y=1.1)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.3), fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "visuals" / "radar_model_comparison_larger_font.pdf", bbox_inches="tight")
plt.show()


# 16 — Model Descriptions & Training Details

In [ ]:
# Load training args for all four models
with open(pert_emopair_dir / "training_args.json") as f:
    args_pert_emopair = json.load(f)
with open(pert_emobank_dir / "training_args.json") as f:
    args_pert_emobank = json.load(f)
with open(MODELS_DIR / "emopair_reward_model_training_args_20260310.json") as f:
    args_rw_emopair = json.load(f)
with open(MODELS_DIR / "emobank_reward_model_training_args_20260308.json") as f:
    args_rw_emobank = json.load(f)

desc_rows = []
for name, args, arch, loss_desc in [
    ("PERT-EmoPair", args_pert_emopair,
     "RoBERTa-large → [CLS] → Linear(1024,3)",
     "MSE on V/A/D (A,D use BT scores)"),
    ("PERT-EmoBank", args_pert_emobank,
     "RoBERTa-large → [CLS] → Linear(1024,3)",
     "MSE on V/A/D (gold EmoBank labels)"),
    ("Reward-EmoPair", args_rw_emopair,
     "RoBERTa-large → [CLS] → V_head(1) + AD_head(2)",
     "V: MSE (pointwise) | A,D: BCE (pairwise, human annotations)"),
    ("Reward-EmoBank", args_rw_emobank,
     "RoBERTa-large → [CLS] → V_head(1) + AD_head(2)",
     "V: MSE (pointwise) | A,D: BCE (pairwise, EmoBank-derived)"),
]:
    lr = args.get("learning_rate", args.get("lr"))
    bs = args.get("effective_batch_size", args.get("batch_size"))
    epochs = args.get("num_epochs", args.get("epochs"))
    desc_rows.append({
        "Model": name,
        "Architecture": arch,
        "Loss": loss_desc,
        "LR": lr,
        "Eff. Batch": bs,
        "Epochs": epochs,
    })

desc_df = pd.DataFrame(desc_rows).set_index("Model")
desc_df

# 17 — Export All Results to CSV and Free up the GPU

In [ ]:
# Save all result tables
out_dir = RESULTS_DIR / "visuals"
out_dir.mkdir(exist_ok=True)

valence_df.to_csv(out_dir / "valence_regression_metrics.csv")
bt_df.to_csv(out_dir / "bt_correlation_metrics.csv")
pw_df.to_csv(out_dir / "pairwise_accuracy.csv")
summary.to_csv(out_dir / "model_comparison_summary.csv")
desc_df.to_csv(out_dir / "model_descriptions.csv")

# Save per-text predictions for downstream use
pred_df = test_items[["item_id", "text", "V", "A", "D"]].copy()
pred_df["A_BT"] = gold_A_bt
pred_df["D_BT"] = gold_D_bt
for name in MODEL_NAMES:
    tag = name.replace("-", "_").lower()
    pred_df[f"{tag}_V"] = MODELS[name]["valence"]
    pred_df[f"{tag}_A"] = MODELS[name]["arousal"]
    pred_df[f"{tag}_D"] = MODELS[name]["dominance"]
pred_df.to_csv(out_dir / "all_model_test_predictions.csv", index=False)

print(f"All outputs saved to {out_dir}/")
print("Files:")
for f in sorted(out_dir.glob("*.csv")) + sorted(out_dir.glob("*.pdf")) + sorted(out_dir.glob("*.tex")):
    print(f"  {f.name}")


In [ ]:
import gc
import torch

# Delete all model references
del pert_emopair, pert_emobank, rw_emopair, rw_emobank
del tok_pert_emopair, tok_pert_emobank, tok_rw_emopair, tok_rw_emobank

# Force garbage collection
gc.collect()

# Empty CUDA cache
torch.cuda.empty_cache()

# (Optional) Check remaining memory
print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU Memory cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print("GPU cleared.")

# Results Summary

## What this notebook says about data quality

This notebook compares four RoBERTa-large models on the EmoPair test set (1,000 items; 8,484 test pairs). The comparison crosses training data source (EmoPair vs. EmoBank) with model architecture (PERT regression vs. Reward pairwise). The main question is not just which model wins, but what the pattern of wins says about the newer EmoPair labels relative to the older EmoBank score averages.

A separate result not computed in this notebook is also important: EmoPair shows substantially higher annotator reliability under the paired-comparison framework. That reliability result and the model results below point in the same direction.

| Model | Architecture | Training Data | A/D Supervision |
|---|---|---|---|
| **PERT-EmoPair** | RoBERTa -> Linear(1024, 3) | EmoPair | BT scores |
| **PERT-EmoBank** | RoBERTa -> Linear(1024, 3) | EmoBank | Original 1-5 scores |
| **Reward-EmoPair** | RoBERTa -> V head + AD head | EmoPair | Human-validated pairwise LLM comparisons |
| **Reward-EmoBank** | RoBERTa -> V head + AD head | EmoBank | EmoBank-derived pairwise labels |

## 1. Valence is mostly a calibration tie

Valence is the least controversial dimension in this comparison because both datasets retain a direct score signal.

| Model | MAE ↓ | RMSE ↓ | Pearson $r$ ↑ | Spearman $\rho$ ↑ |
|---|---|---|---|---|
| **PERT-EmoBank** | **0.1442** | **0.1948** | 0.8326 | **0.8169** |
| PERT-EmoPair | 0.1527 | 0.2019 | **0.8338** | 0.8163 |
| Reward-EmoPair | 0.1840 | 0.2375 | 0.8079 | 0.7873 |
| Reward-EmoBank | 0.2023 | 0.2580 | 0.7838 | 0.7822 |

Key interpretation:
- PERT-EmoBank has the lowest absolute error, and the MAE advantage over PERT-EmoPair is significant (Wilcoxon $p = 2.59 \times 10^{-4}$).
- But the correlation gap is effectively zero: PERT-EmoPair has Pearson $r = 0.8338$ vs. 0.8326 for PERT-EmoBank, with $\Delta r = 0.0011$ and a bootstrap CI that crosses zero.
- So valence says that EmoBank still provides a useful calibration target, but it does not establish a meaningfully stronger underlying signal than EmoPair.

## 2. EmoPair dominates on EmoPair-defined Arousal and Dominance targets

For Arousal and Dominance, the BT targets are derived from EmoPair pairwise judgments, so this is the most direct in-notebook test of whether the paired-comparison framework yields better supervision.

| Model | A BT Pearson $r$ ↑ | A BT Spearman $\rho$ ↑ | D BT Pearson $r$ ↑ | D BT Spearman $\rho$ ↑ |
|---|---|---|---|---|
| **Reward-EmoPair** | **0.8706** | **0.8774** | 0.7263 | 0.7285 |
| PERT-EmoPair | 0.8616 | 0.8634 | **0.7313** | **0.7334** |
| Reward-EmoBank | 0.6273 | 0.6352 | 0.3177 | 0.3261 |
| PERT-EmoBank | 0.6144 | 0.6638 | 0.4423 | 0.4338 |

Key interpretation:
- The data-source gap is much larger than the architecture gap. Both EmoPair-trained models are far ahead of both EmoBank-trained models on Arousal and Dominance BT correlation.
- The top two models are statistically tied within EmoPair: Arousal $\Delta r = 0.0090$, 95% CI [-0.0032, 0.0216]; Dominance $\Delta r = 0.0050$, 95% CI [-0.0135, 0.0236].
- That pattern strongly suggests the advantage is in the EmoPair labels, not in one special model head.

## 3. EmoBank wins only when judged against the old EmoBank score scale

When evaluation switches to the original 1-5 annotator averages, the ranking reverses.

| Model | A orig Pearson $r$ ↑ | A orig Spearman $\rho$ ↑ | D orig Pearson $r$ ↑ | D orig Spearman $\rho$ ↑ |
|---|---|---|---|---|
| **PERT-EmoBank** | **0.5898** | **0.5017** | **0.5317** | **0.4622** |
| Reward-EmoBank | 0.4853 | 0.4316 | 0.3898 | 0.3471 |
| Reward-EmoPair | 0.4755 | 0.4540 | 0.3604 | 0.3217 |
| PERT-EmoPair | 0.4663 | 0.4444 | 0.3447 | 0.3167 |

Key interpretation:
- PERT-EmoBank is best at reproducing the older EmoBank-style score scale for Arousal and Dominance.
- Its advantage over Reward-EmoBank is significant: Arousal $\Delta r = 0.1042$, 95% CI [0.0729, 0.1363]; Dominance $\Delta r = 0.1416$, 95% CI [0.0910, 0.1926].
- This does not mean EmoBank is the better supervision source for comparative judgments. It means EmoBank-trained models are better calibrated to the legacy scale that EmoBank itself defines.

## 4. Pairwise accuracy is the cleanest behavioral test

Pairwise accuracy asks whether the model agrees with held-out comparison judgments. That is the closest match to the annotation task EmoPair was designed to support.

| Model | Valence Acc ↑ | Arousal Acc ↑ | Dominance Acc ↑ |
|---|---|---|---|
| **PERT-EmoBank** | **84.79%** | 73.99% | 64.93% |
| PERT-EmoPair | 84.62% | 84.34% | 76.96% |
| **Reward-EmoPair** | 83.13% | **85.37%** | **77.03%** |
| Reward-EmoBank | 82.68% | 72.82% | 61.33% |

Key interpretation:
- Valence is again close across models, which is why it is not the main evidence in the data-quality argument.
- For Arousal and Dominance, the EmoPair-trained models are dramatically better on held-out comparisons.
- Dominance is especially revealing: EmoPair-trained models are around 77% accurate, while EmoBank-trained models fall to 65% and 61%, much closer to chance than to the EmoPair systems.

## 5. Statistical summary

| Comparison | Metric | Result | Interpretation |
|---|---|---|---|
| PERT-EmoBank vs. PERT-EmoPair | Valence MAE | Wilcoxon $p = 2.59 \times 10^{-4}$ | PERT-EmoBank has lower absolute valence error |
| PERT-EmoPair vs. PERT-EmoBank | Valence Pearson $r$ | $\Delta r = 0.0011$, CI [-0.0125, 0.0147] | No meaningful ranking difference |
| Reward-EmoPair vs. PERT-EmoPair | Arousal BT Pearson $r$ | $\Delta r = 0.0090$, CI [-0.0032, 0.0216] | No meaningful architecture difference within EmoPair |
| PERT-EmoPair vs. Reward-EmoPair | Dominance BT Pearson $r$ | $\Delta r = 0.0050$, CI [-0.0135, 0.0236] | No meaningful architecture difference within EmoPair |
| PERT-EmoBank vs. Reward-EmoBank | Arousal orig Pearson $r$ | $\Delta r = 0.1042$, CI [0.0729, 0.1363] | Strong advantage for the EmoBank-trained regressor on the legacy scale |
| PERT-EmoBank vs. Reward-EmoBank | Dominance orig Pearson $r$ | $\Delta r = 0.1416$, CI [0.0910, 0.1926] | Strong advantage for the EmoBank-trained regressor on the legacy scale |

## Bottom line on EmoPair vs. EmoBank data quality

1. **For comparative affect judgments, EmoPair looks like the higher-quality supervision source.** Two different model families trained on EmoPair both beat both EmoBank-trained models on BT correlation and held-out pairwise accuracy, especially for Arousal and Dominance.
2. **The strongest evidence is cross-architecture consistency.** Within EmoPair, PERT and Reward are statistically tied on the main BT metrics. Across datasets, the gaps are large. That is the signature of a data effect more than an architecture effect.
3. **EmoBank still contains useful signal, but it serves a different purpose.** It remains strong for Valence and for reproducing the historical 1-5 score scale. That is a calibration advantage, not evidence that the labels are better for comparative emotional judgments.
4. **Taken together with the separate reliability result, the most defensible interpretation is that EmoPair is the cleaner measurement framework.** The paired-comparison design appears to reduce annotator scale-use noise and produce labels that are more internally consistent and more behaviorally faithful on held-out comparisons.
5. **So the answer is not that EmoBank is useless.** The answer is that EmoPair is better aligned with the judgments we actually want to model, while EmoBank is mainly valuable for backward compatibility with the older direct-score paradigm.

## Publication-Style Discussion Paragraph

Across two model families, the dominant pattern was that models trained on EmoPair generalized substantially better to held-out comparative judgments than models trained on EmoBank, particularly for Arousal and Dominance. Because this advantage held across both a regression formulation and a pairwise reward formulation, while the top two EmoPair models were statistically indistinguishable on the main Bradley-Terry metrics, the most plausible explanation is a difference in supervision quality rather than architecture. When taken together with the separate evidence that EmoPair yields higher annotator reliability under a paired-comparison protocol, these results suggest that EmoPair provides a cleaner and more behaviorally faithful measurement of comparative affect than the older EmoBank score averages. EmoBank remains useful where backward compatibility with legacy 1--5 scale judgments is required, but the present evidence indicates that EmoPair is the stronger resource for modeling relative emotional judgments.

## Potential Paper Text 

We recommend EmoPair as the preferred resource for modeling comparative affect in text. Across both regression and reward-model architectures, models trained on EmoPair substantially outperformed models trained on EmoBank on Bradley-Terry correlations and held-out pairwise accuracy for Arousal and Dominance, while separate reliability analyses indicate higher annotator consistency under the paired-comparison framework. Taken together, these findings suggest that EmoPair provides cleaner supervision and a more behaviorally faithful representation of comparative emotional judgments than older direct-score averages.

For applied use, we recommend PERT-EmoPair as the default deployable model when a single model is needed to assign a per-text affect profile, because it offers the most balanced performance across Valence, Arousal, and Dominance and is easier to interpret as a single-text estimator. Reward-EmoPair is a strong alternative when the primary task is ranking or comparing texts, especially for Arousal and Dominance. In either case, outputs should be interpreted primarily as relative affect estimates for texts similar to the EmoPair domain, rather than as universally calibrated absolute measurements of emotional intensity.